# ⚡ ESD Risk Prediction — Full Pipeline Notebook
### Explainable Attention-Based Situational Awareness Framework for Wearable IoT Devices

**Team roles covered:**
- 🔵 Member 1: Synthetic dataset generation (ESD physics simulation)
- 🟣 Member 2: Attention-based deep learning model (YOU)
- 🔴 Member 3: XAI — SHAP saliency + attention maps + Knowledge Graph

---
**Run this on Google Colab:** Runtime → Run all

In [ ]:
# ── STEP 0: Install dependencies ─────────────────────────────
!pip install torch torchvision scikit-learn pandas numpy scipy matplotlib seaborn plotly streamlit -q
print('✓ Dependencies installed')

In [ ]:
# ── STEP 1: Clone / set up project structure ─────────────────
import os, sys
from pathlib import Path

# If running in Colab, clone or upload the project files.
# Otherwise set ROOT to the local project path.
try:
    import google.colab
    IN_COLAB = True
    ROOT = Path('/content/esd_project')
    if not ROOT.exists():
        (ROOT / 'data').mkdir(parents=True, exist_ok=True)
        (ROOT / 'models').mkdir(parents=True, exist_ok=True)
        (ROOT / 'xai').mkdir(parents=True, exist_ok=True)
        (ROOT / 'dashboard').mkdir(parents=True, exist_ok=True)
        print('⚠ Colab detected. Please upload the project .py files or mount Drive.')
        print('  Expected structure:')
        print('    /content/esd_project/data/generate_dataset.py')
        print('    /content/esd_project/models/attention_model.py')
        print('    /content/esd_project/models/train.py')
        print('    /content/esd_project/xai/explainer.py')
except ImportError:
    IN_COLAB = False
    ROOT = Path.cwd().resolve()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    while ROOT.parent != ROOT and not ((ROOT / 'data').exists() and (ROOT / 'models').exists()):
        ROOT = ROOT.parent
    if not ((ROOT / 'data').exists() and (ROOT / 'models').exists()):
        raise FileNotFoundError('Could not locate the esd_project root from the current notebook directory')

ROOT = str(ROOT)
sys.path.insert(0, ROOT)
print(f'✓ Project root: {ROOT}')

## 🔵 Member 1 — Dataset Generation

In [ ]:
# ── STEP 2: Generate synthetic ESD dataset ────────────────────
import runpy
dataset_globals = runpy.run_path(os.path.join(ROOT, 'data', 'generate_dataset.py'))
OUTPUT_PATH = dataset_globals['OUTPUT_PATH']
print('\n✓ Dataset saved to:', OUTPUT_PATH)

In [ ]:
# ── STEP 3: Explore the dataset ───────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

df = pd.read_csv(os.path.join(ROOT, 'data', 'esd_dataset.csv'))
print(df.describe().round(2).to_string())
print(f'\nShape: {df.shape}')
print('\nRisk label distribution:')
print(df['risk_label'].value_counts().sort_index().rename({0:'Low',1:'Medium',2:'High'}))

In [ ]:
# ── STEP 4: Visualise sensor correlations with ESD risk ───────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sensors = ['humidity_pct', 'temperature_c', 'efield_vm', 'contact_voltage_v', 'movement_g', 'risk_score']
colors  = ['#2ecc71', '#e67e22', '#e74c3c', '#9b59b6', '#3498db', '#f1c40f']
titles  = ['Humidity (%)', 'Temperature (°C)', 'E-Field (V/m)', 'Contact Voltage (V)', 'Movement (g)', 'Risk Score']

for ax, col, color, title in zip(axes.flat, sensors, colors, titles):
    sample = df[col].iloc[:1000].values
    ax.plot(sample, color=color, linewidth=0.8, alpha=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Time steps')
    ax.spines[['top','right']].set_visible(False)
    ax.set_facecolor('#f8f9fa')

plt.suptitle('ESD Wearable Sensor Signals (first 1000 samples)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'xai', 'sensor_overview.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Sensor overview plot saved')

In [ ]:
# ── STEP 5: Correlation heatmap ───────────────────────────────
import seaborn as sns

num_cols = ['humidity_pct','temperature_c','efield_vm','contact_voltage_v','movement_g','risk_score']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=ax, square=True, linewidths=0.5,
            xticklabels=['Humid','Temp','E-Field','Voltage','Movement','Risk'],
            yticklabels=['Humid','Temp','E-Field','Voltage','Movement','Risk'])
ax.set_title('Sensor Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'xai', 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Correlation heatmap saved')

## 🟣 Member 2 — Attention Model Training

In [ ]:
# ── STEP 6: Verify model architecture ────────────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
while ROOT.parent != ROOT and not (ROOT / 'models').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from models.attention_model import build_model

model = build_model(window_size=50)
print(model)
print(f'\nTotal trainable parameters: {model.count_parameters():,}')

# Test forward pass
B, T = 4, 50
dummy_num  = torch.randn(B, T, 5)
dummy_cats = {
    'activity':    torch.randint(0, 4, (B,)),
    'environment': torch.randint(0, 4, (B,)),
    'fabric_type': torch.randint(0, 4, (B,)),
}
logits, attn = model(dummy_num, dummy_cats, return_attention=True)
print(f'\n✓ Forward pass OK')
print(f'  Input  : ({B}, {T}, 5)')
print(f'  Output : {logits.shape}  → 3 risk classes')
print(f'  Attn   : {len(attn)} layers × {attn[0].shape}')

In [ ]:
# ── STEP 7: Train the model ───────────────────────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
while ROOT.parent != ROOT and not (ROOT / 'models').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.train import train

model, results, scaler, encoders = train(epochs=30)
print('\n✓ Training complete')

In [ ]:
# ── STEP 8: Plot training history ────────────────────────────
history = results['history']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], color='#e74c3c', linewidth=2, label='Train Loss')
ax1.set_title('Training Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.spines[['top','right']].set_visible(False)

ax2.plot(history['val_f1'], color='#2ecc71', linewidth=2, label='Val Macro F1')
ax2.plot(history['train_acc'], color='#3498db', linewidth=2, label='Train Accuracy', linestyle='--')
ax2.set_title('Validation F1 & Train Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score')
ax2.legend(); ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'xai', 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'\nFinal Test Results:')
print(f'  Accuracy : {results["accuracy"]:.4f}')
print(f'  Macro F1 : {results["f1"]:.4f}')
print(f'  AUC-ROC  : {results["auc"]:.4f}')

In [ ]:
# ── STEP 9: Baseline comparison ───────────────────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
while ROOT.parent != ROOT and not (ROOT / 'models').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.train import prepare_data, run_baselines
_, _, test_loader, _, _, _, test_ds = prepare_data()
baseline_results = run_baselines(test_ds)

# Comparison table
comparison = {
    'Attention Model (Ours)': {'accuracy': results['accuracy'], 'f1': results['f1']},
    **baseline_results
}
comp_df = pd.DataFrame(comparison).T.round(4)
print('\n── Model Comparison Table ──')
print(comp_df.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(comp_df))
bars1 = ax.bar([i-0.2 for i in x], comp_df['accuracy'], 0.35, label='Accuracy', color='#3498db')
bars2 = ax.bar([i+0.2 for i in x], comp_df['f1'], 0.35, label='Macro F1', color='#2ecc71')
ax.set_xticks(list(x)); ax.set_xticklabels(comp_df.index, rotation=15, ha='right')
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Model Comparison: Accuracy & Macro F1', fontweight='bold')
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'xai', 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🔴 Member 3 — XAI Analysis

In [ ]:
# ── STEP 10: Feature importance (gradient saliency) ───────────
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
while ROOT.parent != ROOT and not (ROOT / 'models').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from xai.explainer import compare_explainer_runtime, plot_global_importance
from models.train import prepare_data, DEVICE

train_loader, val_loader, test_loader, _, _, _, _ = prepare_data()
model = model.to(DEVICE)

runtime = compare_explainer_runtime(model, test_loader, DEVICE, n_batches=20)
importance = runtime['gradient_importance']

print('Explainer Runtime:')
print(f"  GradientExplainer : {runtime['gradient_seconds']:.2f}s")
print(f"  SHAPExplainer     : {runtime['shap_seconds']:.2f}s")
print(f"  SHAP / Gradient   : {runtime['speedup']:.1f}x slower")


print('Global Feature Importance:')
for feat, imp in zip(['Humidity','Temperature','E-Field','Voltage','Movement'], importance):
    bar = '█' * int(imp / importance.max() * 20)
    print(f'  {feat:20s} {bar} {imp:.4f}')

path = plot_global_importance(importance)
from IPython.display import Image
Image(path)

In [ ]:
# ── STEP 11: Attention map for one sample ─────────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
while ROOT.parent != ROOT and not (ROOT / 'models').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from xai.explainer import plot_attention_map

# Pick one test sample
sample_num, sample_cats, sample_lbl = next(iter(test_loader))
idx = 0
single_num  = sample_num[idx:idx+1]
single_cats = {k: v[idx:idx+1] for k, v in sample_cats.items()}

path = plot_attention_map(model, single_num, single_cats, DEVICE)
Image(path)

In [ ]:
# ── STEP 12: Plain-English explanation ───────────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
while ROOT.parent != ROOT and not (ROOT / 'models').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from xai.explainer import generate_explanation, RISK_LABELS
import torch.nn.functional as F

model.eval()
with torch.no_grad():
    logits = model(single_num.to(DEVICE), {k: v.to(DEVICE) for k, v in single_cats.items()})
    probs  = F.softmax(logits, dim=-1).squeeze().cpu().numpy()

pred_class = int(probs.argmax())

# Use last raw sensor reading as context
df_sample = pd.read_csv(os.path.join(ROOT, 'data', 'esd_dataset.csv'))
sample_row = df_sample.iloc[5000]
sensor_ctx = {
    'humidity_pct':      sample_row['humidity_pct'],
    'temperature_c':     sample_row['temperature_c'],
    'efield_vm':         sample_row['efield_vm'],
    'contact_voltage_v': sample_row['contact_voltage_v'],
    'movement_g':        sample_row['movement_g'],
    'activity':          sample_row['activity'],
    'environment':       sample_row['environment'],
    'fabric_type':       sample_row['fabric_type'],
}

explanation = generate_explanation(sensor_ctx, pred_class, probs)
print(explanation)

In [ ]:
# ── STEP 13: Knowledge Graph visualisation ───────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
while ROOT.parent != ROOT and not (ROOT / 'models').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from xai.explainer import get_knowledge_graph
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

kg = get_knowledge_graph()

# Simple force-directed layout approximation
positions = {
    'ESD_Risk':            (0.5,  0.5),
    'Humidity':            (0.1,  0.8),
    'ElectricField':       (0.2,  0.3),
    'ContactVoltage':      (0.8,  0.3),
    'Movement':            (0.9,  0.7),
    'Polyester':           (0.95, 0.5),
    'Wool':                (0.85, 0.15),
    'Synthetic':           (0.75, 0.08),
    'Cotton':              (0.65, 0.15),
    'Lab':                 (0.3,  0.9),
    'Office':              (0.15, 0.65),
    'HandlingElectronics': (0.5,  0.85),
    'WristStrap':          (0.1,  0.35),
    'Grounding':           (0.2,  0.15),
    'HumidityControl':     (0.4,  0.1),
}

type_colors = {'concept':'#e74c3c','sensor':'#3498db','material':'#9b59b6',
               'environment':'#2ecc71','activity':'#f39c12','mitigation':'#1abc9c'}

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_facecolor('#1a1a2e')
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.axis('off')

# Draw edges
for edge in kg['edges']:
    x0, y0 = positions[edge['from']]
    x1, y1 = positions[edge['to']]
    color = '#e74c3c' if edge['weight'] > 0 else '#2ecc71'
    alpha = min(0.9, abs(edge['weight']) * 1.2)
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
        arrowprops=dict(arrowstyle='->', color=color, alpha=alpha, lw=1.5))
    mx, my = (x0+x1)/2, (y0+y1)/2
    ax.text(mx, my, edge['relation'].replace('_',' '), fontsize=5,
            color='#aaaaaa', ha='center', va='center', alpha=0.7)

# Draw nodes
for node in kg['nodes']:
    x, y = positions[node['id']]
    color = type_colors.get(node['type'], '#888')
    size  = 800 if node['id'] == 'ESD_Risk' else 400
    ax.scatter(x, y, s=size, c=color, zorder=5, edgecolors='white', linewidths=1.5)
    ax.text(x, y-0.05, node['id'].replace('_',' '), fontsize=7,
            ha='center', va='top', color='white', fontweight='bold')

# Legend
patches = [mpatches.Patch(color=v, label=k) for k, v in type_colors.items()]
ax.legend(handles=patches, loc='lower left', fontsize=8,
          facecolor='#2d2d44', labelcolor='white', edgecolor='#555')

ax.set_title('ESD Knowledge Graph — Risk Reasoning Network',
             color='white', fontsize=13, fontweight='bold', pad=15)
fig.patch.set_facecolor('#1a1a2e')
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'xai', 'knowledge_graph.png'), dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('✓ Knowledge Graph saved')

In [ ]:
# ── STEP 14: Launch dashboard ────────────────────────────────
print('To launch the real-time dashboard, run:')
print('  streamlit run dashboard/app.py')
print('')
print('In Google Colab, use:')
print('  !pip install streamlit pyngrok -q')
print('  from pyngrok import ngrok')
print('  !streamlit run dashboard/app.py &')
print('  ngrok.connect(8501)')
print('')
print('=' * 55)
print('  ✅ ALL PIPELINE STEPS COMPLETE')
print('=' * 55)
print('  Outputs saved:')
import os, glob
for f in sorted(glob.glob(os.path.join(ROOT, 'xai', '*.png'))):
    print(f'    {f}')
print(f'    {os.path.join(ROOT, "models", "best_model.pt")}')
print(f'    {os.path.join(ROOT, "data", "esd_dataset.csv")}')